## Ideal MHD Stability with TokaMaker + GPEC

This noteboook demonstrates some of the key ideal MHD capabilities of GPEC by coupling it to the TokaMaker Grad-Shafranov tool. The example shown will start with an equilibrium (geqdsk file) and kinetic profiles (kin file) for DIII-D shot 147131. We will first set up the TokaMaker equilibrium solver and then choose our settings for GPEC. We will utilize GPEC to calculate the following: 

1. External kink stability (ideal $\beta_N$ limit search)
2. Ideal tearing stability
3. Glasser correction to ideal tearing stability

Each will be plotted over a scan of several values of betaN.


In this example, we will use the Python package juliacall to make calls to GPEC, which is written in Julia. Before making any imports, juliacall requires that we set its environment variables 'PYTHON_JULIACALL_EXE' and 'PYTHON_JULIACALL_PROJECT'. **Make sure to point the `PYTHON_JULIACALL_PROJECT` variable to the location of your local copy of GPEC.**



In [ ]:
import os
import subprocess
os.environ['PYTHON_JULIACALL_EXE'] = subprocess.run(['which','julia'],capture_output=True,text=True).stdout.strip()
os.environ['PYTHON_JULIACALL_PROJECT'] = "/path/to/GPEC" # point to the location of your local GPEC

from juliacall import Main as jl

import sys
import shutil
import h5py
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Below, we search your local machine for the location of your copy of OpenFUSIONToolkit:

In [ ]:
def find_oft():
    env = os.environ.get("OFT_ROOTPATH")
    if env and (Path(env) / "python" / "OpenFUSIONToolkit" / "__init__.py").is_file():
        return Path(env)
    for match in Path.home().rglob("python/OpenFUSIONToolkit"):
        if (match / "__init__.py").is_file():   # skip bytecode-cache mirrors (e.g. macOS ~/Library/Caches)
            return match.parent.parent           # match = .../<oft_root>/python/OpenFUSIONToolkit
    raise RuntimeError(
        "Could not locate OpenFUSIONToolkit. Point your OFT_ROOTPATH environment "
        "variable to your OFT build directory (the one containing python/OpenFUSIONToolkit)."
    )

In [ ]:
oft_root = find_oft()

In [ ]:
oft_py = os.path.join(oft_root, "python")
sys.path.insert(0, oft_py)

import OpenFUSIONToolkit as _oft_pkg
from OpenFUSIONToolkit import OFT_env
from OpenFUSIONToolkit.TokaMaker import TokaMaker
from OpenFUSIONToolkit.TokaMaker.meshing import load_gs_mesh
from OpenFUSIONToolkit.TokaMaker.util import read_eqdsk

Next we read in the relevant data from our sample eqdsk, kinetic EFIT, and the DIII-D mesh file:

In [ ]:
eqdsk_path = "g147131.02300_DIIID_KEFIT"
eqdsk = read_eqdsk(eqdsk_path)

def read_kinetic_file(path):
    df = pd.read_csv(path, sep=r"\s+")
    return df.to_dict(orient="list")
kinetic_file = read_kinetic_file("g147131.02300_DIIID_KEFIT_corrected.kin")

mesh_file = "../../DIIID/DIIID_mesh.h5"

working_directory = Path(os.getcwd()) / "gpec_runs"

This cell sets up the enviornmet for OpenFUSIONToolkit to run. Note that this cell may only be executed once per session or it will give an error. 

In [ ]:
myOFT = OFT_env(nthreads=2)
mygs  = TokaMaker(myOFT)

Here and below we set up TokaMaker - reading in the mesh, defining target values for Ip and on-axis pressure, defining the isoflux points, and setting P' and FF'.

In [ ]:
mesh_pts, mesh_lc, mesh_reg, coil_dict, cond_dict = load_gs_mesh(str(mesh_file))
mygs.setup_mesh(mesh_pts, mesh_lc, mesh_reg)
mygs.setup_regions(cond_dict=cond_dict, coil_dict=coil_dict)
mygs.setup(order=2, F0=abs(eqdsk["rcentr"] * eqdsk["bcentr"]))

mygs.set_coil_vsc({"F9A": 1.0, "F9B": -1.0}) # is this necessary? 

In [ ]:
mygs.set_targets(Ip=abs(eqdsk["ip"]), pax=eqdsk["pres"][0])

isoflux_pts = eqdsk["rzout"].copy()
weights     = np.ones(isoflux_pts.shape[0]) * 50.0
mygs.set_isoflux_constraints(isoflux_pts, weights=weights)

coil_names = [
    "ECOILA", "ECOILB",
    "F1A", "F2A", "F3A", "F4A", "F5A", "F6A", "F7A", "F8A", "F9A",
    "F1B", "F2B", "F3B", "F4B", "F5B", "F6B", "F7B", "F8B", "F9B",
]
reg_terms = [mygs.coil_reg_term({c: 1.0}, target=0.0, weight=1.0) for c in coil_names]
reg_terms.append(mygs.coil_reg_term({"#VSC": 1.0}, target=0.0, weight=1.0e-2))
mygs.set_coil_reg(reg_terms=reg_terms)

In [ ]:
psi_prof = np.array(kinetic_file["psi"])
kinetic_file_nr = len(psi_prof)

ffprim = eqdsk["ffprim"]
pprime = eqdsk["pprime"]
psi_eqdsk = np.linspace(0.0, 1.0, np.size(ffprim))

ffp_interp = np.interp(psi_prof, psi_eqdsk, ffprim)
pp_interp  = np.interp(psi_prof, psi_eqdsk, pprime)

mygs.set_profiles(
    ffp_prof={"type": "linterp", "y": ffp_interp, "x": psi_prof},
    pp_prof ={"type": "linterp", "y": pp_interp,  "x": psi_prof},
)

err_flag = mygs.init_psi(1.7, 0.0, 0.5, 1.3, -0.4) # what is this 
err_flag = mygs.solve()
mygs.print_info()

This is our import of the GPEC Julia package (GeneralizedPerturbedEquilibrium). "using" is the Julia equivalent of Python's "import", and jl.seval() is the juliacall function that allows us to directly call Julia code from Python. 

In [ ]:
jl.seval("using GeneralizedPerturbedEquilibrium")

## Ideal kink and tearing stability

Now that the environment is fully set up and TokaMaker has been intialized, we will build the machinery to analyze the ideal kink and tearing stability for this case. 

For this case we will look at the no-wall approximation, but it is also possible to run with a wall by changing "shape" from "nowall" to "wall". Some default wall distance parameters are already included (only used when shape = wall). 

In [ ]:
# GPEC Settings 
n_tor = 1 # global kink 

kinetic = {
    "kinetic_source": "fixed",
    "kinetic_factor": 0.0,
}

wall = {
    "shape": "nowall", # change this to "wall" to run with a resistive wall 
    "a":     0.10,       
    "aw":    0.05,
    "bw":    1.5,
    "cw":    0.0,
    "dw":    0.5,
    "tw":    0.05,
    "equal_arc_wall": True,
}

GPEC runs by writing out a .toml file containing all the settings for the run. The block below writes the settings that we selected in the cells above into the correct format for GPEC to read.

In [ ]:
def make_gpec_toml(eq_filename, n_tor, wall, kinetic, output_h5="gpec.h5"):
    wall_block = (
        '[Wall]\nshape = "nowall"\n'
        if wall["shape"] == "nowall"
        else "[Wall]\n"
        + f'shape = "{wall["shape"]}"\n'
        + f'a = {wall["a"]}\n'
        + "\n".join(f"{k} = {wall[k]}" for k in ("aw", "bw", "cw", "dw", "tw") if k in wall)
    )
    return f"""
[Equilibrium]
eq_filename = "{eq_filename}"
eq_type     = "efit"
jac_type    = "hamada"
grid_type   = "ldp"
psilow      = 1e-4
psihigh     = 0.995
mpsi        = 257
mtheta      = 514
 
{wall_block}
 
[ForceFreeStates]
local_stability_flag = true
vac_flag = true
set_psilim_via_dmlim = true
dmlim = 0.2
qhigh = 1e3
qlow  = 1.02
nn_low  = {n_tor}
nn_high = {n_tor}
kinetic_source = "{kinetic["kinetic_source"]}"
kinetic_factor = {kinetic["kinetic_factor"]}
delta_mlow  = 8
delta_mhigh = 8
mthvac      = 960
 
HDF5_filename = "{output_h5}"
"""

Below, we define the file I/O process for GPEC. This includes: 
1. Defining a `run_dir` to which the TokaMaker geqdsk files and the GPEC toml files for each run will be written 
2. Reading out the quantities of interest from the resulting gpec.h5 file 

This function will be called in a loop for increasing values of $\beta_N$. 

In [ ]:
jl.seval(
    "run_gpec_main(path::AbstractString) = GeneralizedPerturbedEquilibrium.main([path])"
)

def run_gpec(run_dir, geqdsk_path, n_tor, wall, kinetic, output_h5="gpec.h5"):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
 
    eq_dst = run_dir / Path(geqdsk_path).name
    if Path(geqdsk_path).resolve() != eq_dst.resolve():
        shutil.copy2(geqdsk_path, eq_dst)
    (run_dir / "gpec.toml").write_text(
        make_gpec_toml(eq_dst.name, n_tor, wall, kinetic, output_h5)
    )
 
    jl.run_gpec_main(str(run_dir))

    h5_path = run_dir / output_h5
 
    with h5py.File(h5_path, "r") as f:
        et = f["ForceFreeStates/FreeBoundaryStability/eigenmode_energies"][...]
        ep = f["ForceFreeStates/FreeBoundaryStability/eigenmode_plasma_energies"][...]
        ev = f["ForceFreeStates/FreeBoundaryStability/eigenmode_vacuum_energies"][...]
        dp = f["SingularSurfaces/Delta_prime_matrix"][...]

        msing = int(f["SingularSurfaces/rational_count"][()])
        q0    = float(f["Equilibrium/q_axis"][()])
        qmax  = float(f["Equilibrium/q_max"][()]) if "Equilibrium/q_max" in f else float(f["Info/qlim"][()])

    def as_complex(a):
        if a.dtype.names and {"r", "i"}.issubset(a.dtype.names):
            return a["r"] + 1j * a["i"]
        return np.asarray(a, dtype=complex)

    et_c = as_complex(et)
 
    return {
        "dW_total":  float(np.real(et_c.flat[0])),
        "dW_plasma": float(np.real(as_complex(ep).flat[0])),
        "dW_vacuum": float(np.real(as_complex(ev).flat[0])),
        "et_full":   et_c,
        "q0":        q0,
        "qmax":      qmax,
        "msing":     msing,
        "run_dir":   str(run_dir),
        "h5":        str(h5_path)
    }

In order to scan over values of $\beta_N$, we need to scale up the plasma pressure. Changing the plasma pressure requires a re-solve of the Grad-Shafranov equation - i.e. another call to TokaMaker to solve the equilibrium at the new pressure before feeding the equilibrium into GPEC.  

This cell contains the logic for scaling the pressure by a user-defined fraction, calculating the resulting $P'$, and calling to TokaMaker to solve. The function returns the selected scaling fraction, the pressure and temperature profiles, and $\beta_N$. 

In [ ]:
def rescale_pressure(fraction, n_iterations=3):
    boltzmann_constant = 1.6022e-19
    mu0 = 4.0e-7 * np.pi

    kinetic_file_nr = len(np.array(kinetic_file["psi"]))
    psi_tkmkr, f_bl, fp_bl, _, pp_bl = mygs.get_profiles(npsi=kinetic_file_nr)

    _, _, ravgs_bl, *_           = mygs.get_q(npsi=kinetic_file_nr, psi_pad=0.001)
    R_avg, oneOverR_avg = ravgs_bl['<R>'], ravgs_bl['<1/R>']
    jphi_baseline = R_avg * pp_bl + oneOverR_avg * (f_bl * fp_bl) / mu0

    ne = np.array(kinetic_file["ne(m^-3)"])
    te = np.array(kinetic_file["te(eV)"])

    ni = np.array(kinetic_file["ni(m^-3)"])
    ti = np.array(kinetic_file["ti(eV)"])

    pressure = fraction * boltzmann_constant * (ne * te + ni * ti)

    for _ in range(n_iterations):
        pprime_new = np.gradient(pressure) / (np.gradient(psi_tkmkr) * (mygs.psi_bounds[1] - mygs.psi_bounds[0]))

        mygs.set_targets(Ip = abs(eqdsk["ip"]), pax = pressure[0])

        mygs.set_profiles(
            pp_prof  ={"type": "linterp",      "y": pprime_new,    "x": psi_tkmkr},
            ffp_prof ={"type": "jphi-linterp", "y": jphi_baseline, "x": psi_prof},
        )

        try:
            _eq, nl_its = mygs.solve(return_its=True)
        except ValueError as exc:
            raise RuntimeError(
                f"TokaMaker did not converge at frac={fraction}: {exc}") from exc

    stats = mygs.get_stats()
    return {"p_axis": pressure[0], "beta_n": stats.get("beta_n"), "beta_t": stats.get("beta_tor")}

Here, we define two more helper functions - `write_geqdsk_for_gpec` and `evaluate_factor`. `write_geqdsk_for_gpec` defines the logic for writing out the solved TokaMaker geqdsks for each rescaled pressure value, and `evaluate_factor` wraps each of the aforementioned functions to run the full workflow for a single pressure point.  

In [ ]:
def write_geqdsk_for_gpec(run_dir, tag, nr=257, nz=257):
    run_dir = Path(run_dir); run_dir.mkdir(parents=True, exist_ok=True)
    out = run_dir / f"tkmkr_{tag}.geqdsk"
    mygs.save_eqdsk(str(out), nr=nr, nz=nz, truncate_eq=False, lcfs_pad=0.001)
    return out

def evaluate_factor(fraction, tag=None):
    tag = tag if tag is not None else f"f{fraction:.4f}".replace(".", "p")
    run_dir = working_directory / tag

    tkm = rescale_pressure(fraction)
    geq = write_geqdsk_for_gpec(run_dir, tag)
    gp  = run_gpec(run_dir, geq, n_tor, wall, kinetic)

    return {"frac": fraction, "tag": tag,
            "beta_n":  tkm["beta_n"], "beta_t": tkm["beta_t"],
            "p_axis":  tkm["p_axis"],
            "dW_total": gp["dW_total"], "dW_plasma": gp["dW_plasma"],
            "dW_vacuum": gp["dW_vacuum"],
            "q0":      gp["q0"], "qmax": gp["qmax"], "msing": gp["msing"],
            "run_dir": gp["run_dir"]}


Here the user defines a set of `fractions`, which are the values by which the pressure will be scaled. Then the `fractions` are looped over, running the full loop of pressure rescale -> solve TokaMaker -> run GPEC for each point. In case TokaMaker doesn't converge over the default number of iterations `n_iterations` (3), the TokaMaker solve is re-tried with 3 more iteration. 

In [ ]:
fractions = np.linspace(1.0, 1.8, 8)

scan_results = []
for fraction in fractions: 
    try: 
        result = evaluate_factor(fraction)
    except Exception as exc:
        print(f"{fraction} failed on first attempt: {exc!r}")
        try:
            result = evaluate_factor(fraction, tag=f"f{fraction:.4f}".replace(".", "p") + "_retry")
        except Exception as exc2:
            print(f"{fraction} failed on retry too: {exc2!r}")
            continue
    scan_results.append(result)


In [ ]:
scan_df = pd.DataFrame(scan_results)


In [ ]:
sorted_df = scan_df.sort_values("beta_n").reset_index(drop=True)

beta_limit = None
sign_changes = np.where(np.diff(np.sign(sorted_df["dW_total"])) != 0)[0]
if sign_changes.size > 0:
    i = sign_changes[0]
    b0, b1 = sorted_df["beta_n"].iloc[i], sorted_df["beta_n"].iloc[i + 1]
    w0, w1 = sorted_df["dW_total"].iloc[i], sorted_df["dW_total"].iloc[i + 1]
    beta_limit = b0 + (0.0 - w0) * (b1 - b0) / (w1 - w0)

fig, ax = plt.subplots()
ax.plot(sorted_df["beta_n"], sorted_df["dW_total"], "o-")
ax.axhline(0.0, color="gray", linewidth=0.8)

if beta_limit is not None:
    ax.axvline(beta_limit, color="tab:red", linestyle="--", linewidth=1.0)
    ax.annotate(
        f"$\\beta_N^{{lim}} \\approx {beta_limit:.2f}$",
        xy=(beta_limit, 0.0), xytext=(10, 10), textcoords="offset points",
        color="tab:red",
    )

ax.set_xlabel(r"$\beta_N$ [%]")
ax.set_ylabel(r"$\delta W_{total}$")
plt.show()

The ideal portion of GPEC (DCON) that we ran above calculated not only our external kink energy eigenvalue $\delta W$, but also a matrix of the classical tearing stability parameter $\Delta '$ for the toroidal mode number we selected ($n = 1$) for this example and a range of poloidal mode numbers ($m$). 

Here we'll simply read out the classical $\Delta '$ for our chosen `target_m`, `target_n`, set to 2,1 for this example. 



In [ ]:
target_m, target_n = 2, 1   # the 2/1 surface; n_tor above is already 1, matching this n

def read_delta_prime(h5_path, target_m, target_n):
    with h5py.File(h5_path, "r") as f:
        rational_m = f["SingularSurfaces/rational_m"][...].flatten()
        rational_n = f["SingularSurfaces/rational_n"][...].flatten()
        idx = np.where((rational_m == target_m) & (rational_n == target_n))[0]
        if idx.size == 0:
            raise RuntimeError(f"No rational surface at (m={target_m}, n={target_n}) in {h5_path}")
        idx = idx[0]

        dp = f["SingularSurfaces/Delta_prime_matrix"][...]
        if dp.dtype.names and {"r", "i"}.issubset(dp.dtype.names):
            dp = dp["r"] + 1j * dp["i"]
        return float(np.real(dp[idx, idx]))   # diagonal is transpose-invariant

Next, we'll compute the Glasser correction to the classical $\Delta'$ value. The Glasser effect introduces a stabilization to tearing modes in plasmas with good average curvature and high temperature (= high Lundquist number). While the DCON portion of GPEC that we ran above makes ideal calculations in the bulk plasma region, the Glasser correction term must be calculated with resistive effects taken into account. In GPEC, resistive MHD effects are treated by the SLAYER, or SLab inner LAYER, model.

Below, we build a function to set up the inputs SLAYER needs, which includes the plasma rotation `omega`, and then run SLAYER. The whole function is wrapped in a call to `jl.seval()`, the `juliacall` method for writing functions in Julia syntax. 

In [ ]:
jl.seval(
    """
    function run_gpec_glasser(path::AbstractString, target_m::Int, target_n::Int,
                               psi_arr::AbstractVector{<:Real}, ne_arr::AbstractVector{<:Real},
                               te_arr::AbstractVector{<:Real}, ti_arr::AbstractVector{<:Real},
                               omega_arr::AbstractVector{<:Real};
                               chi_perp::Float64=1.0, zeff::Float64=1.5)
        inputs, eq_config, additional_input = GeneralizedPerturbedEquilibrium.build_inputs_from_toml(path)
        out = GeneralizedPerturbedEquilibrium.main_from_inputs(inputs, eq_config, additional_input, path, "glasser-term")
        ffs = out.ffs

        target_sings = filter(s -> target_m in s.m && target_n in s.n, ffs.surfaces)
        isempty(target_sings) && error("run_gpec_glasser: no ($target_m,$target_n) surface found")

        # omega_e/omega_i (electron/ion diamagnetic frequencies) are required by the
        # SLAYERParameters constructor but do not enter dc_tmp at all (dc_type=:rfitzp only
        # uses dr_val/chi_perp/t_e/zeff/tau_ee/rs/R0/sval_r/n) -- offset them from each other
        # only to satisfy the non-degeneracy check.
        omega_i_arr = (-1.0 .* omega_arr) .- 1.0

        profiles = GeneralizedPerturbedEquilibrium.Utilities.KineticProfiles(;
            psi=psi_arr, n_e=ne_arr, T_e=te_arr, T_i=ti_arr,
            omega=omega_arr, omega_e=omega_arr, omega_i=omega_i_arr)

        params = GeneralizedPerturbedEquilibrium.InnerLayer.SLAYER.build_slayer_inputs(
            ffs.equil, target_sings, profiles;
            mu_i=2.0, zeff=zeff, chi_perp=chi_perp, chi_tor=chi_perp, dc_type=:rfitzp)

        p = params[1]
        return (p.dc_tmp, p.dr_val, p.rs, p.R0, p.sval_r, p.eta, p.tau_r)
    end
    """
)

def compute_glasser_term(run_dir, target_m, target_n, fraction, chi_perp=1.0, zeff=1.5):
    psi_arr = np.array(kinetic_file["psi"], dtype=float)
    ne_arr = np.array(kinetic_file["ne(m^-3)"], dtype=float) * fraction  # matches rescale_pressure's density scaling
    te_arr = np.array(kinetic_file["te(eV)"], dtype=float)
    ti_arr = np.array(kinetic_file["ti(eV)"], dtype=float)
    omega_arr = np.array(kinetic_file["wexb(rad/s)"], dtype=float)

    dc_tmp, dr_val, rs, R0, sval_r, eta, tau_r = jl.run_gpec_glasser(
        str(run_dir), target_m, target_n, psi_arr, ne_arr, te_arr, ti_arr, omega_arr,
        chi_perp=chi_perp, zeff=zeff,
    )
    return {
        "delta_crit": float(dc_tmp), "D_R": float(dr_val), "rs": float(rs), "R0": float(R0),
        "sval_r": float(sval_r), "eta": float(eta), "tau_r": float(tau_r),
    }

Finally, we use the two functions we defined above to (1) pull the ideal $\Delta$' (calculated during the same coarse_scan run from which we calculated the $\beta_N$ limit) and (2) compute the Glasser correction term. From these two terms we can calculate $\Delta_{eff}$ = $\Delta$' - $\Delta_{crit}$

In [ ]:
glasser_results = []
for r in scan_results:
    delta_prime = read_delta_prime(r["run_dir"] + "/gpec.h5", target_m, target_n)
    glasser = compute_glasser_term(r["run_dir"], target_m, target_n, r["frac"])

    glasser_results.append({
        "frac": r["frac"], "beta_n": r["beta_n"],
        "delta_prime": delta_prime, "delta_crit": glasser["delta_crit"],
        "delta_eff": delta_prime - glasser["delta_crit"],
        "D_R": glasser["D_R"],
    })

glasser_df = pd.DataFrame(glasser_results)
glasser_df

Finally, all that's left to do is plot the result of the tearing stability scan. This plot shows both $\Delta$' (the ideal portion), and $\Delta_{eff}$. Note that poor behavior near the location of the ideal $\beta_N$ limit is expected, so the x-axis is a normalized metric - $\beta$ / $\beta_{lim}$ - the proximity to the ideal limit we calculated above.

In [ ]:
if beta_limit is None:
    raise RuntimeError("No ideal beta limit found in the dW scan above; cannot normalize by beta_lim.")
beta_frac = glasser_df["beta_n"] / beta_limit
below_lim = beta_frac <= 1.0

fig, ax = plt.subplots()
ax.plot(beta_frac[below_lim], glasser_df["delta_prime"][below_lim], "o--", color="tab:blue",
        label=r"$\Delta'$ (STRIDE self-coupled)")
ax.plot(beta_frac[below_lim], glasser_df["delta_eff"][below_lim], "o-", color="tab:blue",
        label=r"$\Delta_{eff} = \Delta' - \Delta_{crit}$")
ax.axhline(0.0, color="gray", linewidth=0.8)
ax.axvline(1.0, color="tab:red", linestyle="--", linewidth=1.0, label=r"Ideal limit, $\beta = \beta_{lim}$")
ax.set_xlabel(r"$\beta / \beta_{lim}$")
ax.set_ylabel(f"$\\Delta'$ [m/n = {target_m}/{target_n}]")
ax.set_title(f"Glasser stabilization at the {target_m}/{target_n} surface vs. $\\beta / \\beta_{{lim}}$")
ax.legend()
plt.show()